# 02 — Atmospheric simulation and shapelet mode selection

Produces:
* `fig/shapelet_basis.pdf`  — `fig:basis`, the 28 polar shapelet basis functions at
  `bmax=6`, arranged as a pyramid by order $n$, with the score modes marked.
* `fig/atm_coeff_variation.pdf` — `fig:atm`, how each shapelet coefficient responds to
  atmospheric PSF variation. **This is the figure that motivates the score definition.**

Simulation follows `../../atm_shapelet_variation.ipynb`: 1000 realisations of a
two-component Moffat, randomised in FWHM, Moffat $\beta$, and ellipticity. Each
component is point-symmetric, so the population contains atmospheric size/shape
variation and **no** optical asymmetry by construction.

In [1]:
import importlib.util
spec = importlib.util.spec_from_file_location('plot_config', 'plot_config.py')
plot_config = importlib.util.module_from_spec(spec); spec.loader.exec_module(plot_config)
from plot_config import (PALETTE, BMAX, ODD_ORDER, CENTROID_MODES, FIG,
                         MultipleLocator, apply_style)

import numpy as np
import pandas as pd
import galsim
import matplotlib.pyplot as plt

apply_style()

PIXEL_SCALE = 0.2
STAMP = 64
N_PSF = 1000
SEED = 42
N_COEFF = (BMAX + 1) * (BMAX + 2) // 2
print('n coefficients:', N_COEFF, ' score modes:', ODD_ORDER)

n coefficients: 28  score modes: [6, 7, 8, 9, 15, 16, 17, 18, 19, 20]


## Coefficient index table

GalSim stores $b_{pq}$ ordered by increasing $n = p+q$, and within each $n$ from $m = p-q = n$
down to 0 or 1, real part before imaginary part.

In [2]:
rows = []
for n in range(BMAX + 1):
    for q in range(n // 2 + 1):
        p = n - q; m = p - q
        idx = n * (n + 1) // 2 + 2 * q
        if p == q:
            rows.append(dict(idx=idx, n=n, m=m, comp='', label=f'({n},{m})'))
        else:
            rows.append(dict(idx=idx,     n=n, m=m, comp='Re', label=f'({n},{m})'))
            rows.append(dict(idx=idx + 1, n=n, m=m, comp='Im', label=f'({n},{m})'))
coeff = pd.DataFrame(rows).sort_values('idx').reset_index(drop=True)
coeff['odd'] = coeff['m'] % 2 == 1
coeff['in_score'] = coeff['idx'].isin(ODD_ORDER)
coeff['centroid'] = coeff['idx'].isin(CENTROID_MODES)

# Three classes of mode:
#   in_score  : odd m, n >= 3  -> asymmetry, vanishes for any point-symmetric PSF
#   centroid  : n = 1 (m = 1)  -> odd m, but a pure translation of the stamp, not
#                                 an asymmetry; excluded from the score
#   even m    : everything else -> size, ellipticity, and radial profile shape
assert (coeff['in_score'] == (coeff['odd'] & (coeff['n'] >= 3))).all()
assert (coeff['centroid'] == (coeff['n'] == 1)).all()
print(f"score modes  : {coeff.loc[coeff['in_score'],'idx'].tolist()}")
print(f"centroid     : {coeff.loc[coeff['centroid'],'idx'].tolist()}")
print(f"even-m modes : {coeff.loc[~coeff['odd'],'idx'].tolist()}\n")
print(coeff.to_string(index=False))

score modes  : [6, 7, 8, 9, 15, 16, 17, 18, 19, 20]
centroid     : [1, 2]
even-m modes : [0, 3, 4, 5, 10, 11, 12, 13, 14, 21, 22, 23, 24, 25, 26, 27]

 idx  n  m comp label   odd  in_score  centroid
   0  0  0      (0,0) False     False     False
   1  1  1   Re (1,1)  True     False      True
   2  1  1   Im (1,1)  True     False      True
   3  2  2   Re (2,2) False     False     False
   4  2  2   Im (2,2) False     False     False
   5  2  0      (2,0) False     False     False
   6  3  3   Re (3,3)  True      True     False
   7  3  3   Im (3,3)  True      True     False
   8  3  1   Re (3,1)  True      True     False
   9  3  1   Im (3,1)  True      True     False
  10  4  4   Re (4,4) False     False     False
  11  4  4   Im (4,4) False     False     False
  12  4  2   Re (4,2) False     False     False
  13  4  2   Im (4,2) False     False     False
  14  4  0      (4,0) False     False     False
  15  5  5   Re (5,5)  True      True     False
  16  5  5   Im (5,5)  True      

## Figure: shapelet basis functions

In [ ]:
C_SCORE = PALETTE[0]    # burnt orange: modes entering the score
C_CENT  = PALETTE[2]    # slate blue:   n=1 centroid modes (odd m, excluded)
C_EVEN  = '#999999'     # grey:         even m

def mode_color(r):
    if r['in_score']: return C_SCORE
    if r['centroid']: return C_CENT
    return C_EVEN

fig, axes = plt.subplots(BMAX + 1, BMAX + 1, figsize=(6.6, 6.9))
fig.subplots_adjust(left=0.11, right=0.99, top=0.95, bottom=0.07,
                    wspace=0.25, hspace=0.45)
for ax in axes.flat:
    ax.axis('off')

row_left = {}
for n in range(BMAX + 1):
    sub = coeff[coeff['n'] == n].reset_index(drop=True)
    offset = (BMAX + 1 - len(sub)) // 2          # centre the row -> pyramid
    row_left[n] = axes[n, offset]
    for j, r in sub.iterrows():
        ax = axes[n, j + offset]
        c = mode_color(r)
        b = np.zeros(N_COEFF); b[int(r['idx'])] = 1.0
        img = galsim.Image(28, 28, scale=PIXEL_SCALE)
        galsim.Shapelet(0.3, BMAX, b).drawImage(image=img, method='sb')
        v = np.abs(img.array).max()
        ax.imshow(img.array, origin='lower', cmap='RdBu_r', vmin=-v, vmax=v)
        ax.set_xticks([]); ax.set_yticks([]); ax.axis('on')
        for s in ax.spines.values():
            s.set_linewidth(1.8 if r['in_score'] else 0.5)
            s.set_color(c)
        ax.set_title(f"{r['label']}{r['comp']}", fontsize=6, pad=1.5, color=c)

# Row labels: figure-level text so we never touch a panel's frame.
for n, ax0 in row_left.items():
    bb = ax0.get_position()
    fig.text(0.085, bb.y0 + bb.height / 2, f'$n={n}$',
             ha='right', va='center', fontsize=8)

handles = [plt.Line2D([], [], marker='s', ls='', markersize=8, markerfacecolor='none',
                      markeredgewidth=1.8, markeredgecolor=C_SCORE,
                      label='in score: odd $m$, $n\\geq3$'),
           plt.Line2D([], [], marker='s', ls='', markersize=8, markerfacecolor='none',
                      markeredgewidth=1.0, markeredgecolor=C_CENT,
                      label='centroid ($n=1$), excluded'),
           plt.Line2D([], [], marker='s', ls='', markersize=8, markerfacecolor='none',
                      markeredgewidth=1.0, markeredgecolor=C_EVEN,
                      label='even $m$, excluded')]
fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False,
           fontsize=8, bbox_to_anchor=(0.5, 0.0))

fig.savefig(FIG / 'shapelet_basis.pdf')
print('wrote', FIG / 'shapelet_basis.pdf')

## Simulate the atmospheric PSF population

In [4]:
rng = np.random.default_rng(SEED)
fwhm = rng.uniform(0.5, 1.2, N_PSF)
beta = rng.uniform(2.5, 4.5, N_PSF)
g1 = rng.normal(0.0, 0.03, N_PSF)
g2 = rng.normal(0.0, 0.03, N_PSF)
gm = np.hypot(g1, g2); clip = gm > 0.3
g1[clip] = g1[clip] / gm[clip] * 0.3
g2[clip] = g2[clip] / gm[clip] * 0.3

bvec, fwhm_fit, gmod_fit = [], [], []
for i in range(N_PSF):
    psf = (galsim.Moffat(fwhm=fwhm[i], beta=beta[i]).shear(g1=float(g1[i]), g2=float(g2[i]))
           + galsim.Moffat(fwhm=fwhm[-i], beta=beta[-i]).shear(g1=float(g1[-i]), g2=float(g2[-i])))
    img = galsim.Image(STAMP, STAMP, scale=PIXEL_SCALE)
    psf.drawImage(image=img, method='sb')
    try:
        hsm = galsim.hsm.FindAdaptiveMom(img)
        shp = galsim.Shapelet.fit(hsm.moments_sigma * PIXEL_SCALE, BMAX, img, normalization='sb')
    except Exception:
        continue
    bvec.append(shp.bvec.copy())
    fwhm_fit.append(2.3548 * hsm.moments_sigma * PIXEL_SCALE)
    gmod_fit.append(np.hypot(g1[i], g2[i]))

bvec = np.array(bvec); fwhm_fit = np.array(fwhm_fit); gmod_fit = np.array(gmod_fit)
print(f'fitted {len(bvec)} / {N_PSF} PSFs')
print(f'measured FWHM range: {fwhm_fit.min():.2f} - {fwhm_fit.max():.2f} arcsec')

fitted 1000 / 1000 PSFs
measured FWHM range: 0.56 - 1.32 arcsec


In [5]:
total = np.sum(bvec ** 2, axis=1)
coeff['std'] = bvec.std(axis=0)
coeff['medfrac'] = np.median(bvec ** 2 / total[:, None], axis=0)
coeff['r_fwhm'] = [np.corrcoef(bvec[:, i], fwhm_fit)[0, 1] for i in range(N_COEFF)]
coeff['r_g'] = [np.corrcoef(bvec[:, i], gmod_fit)[0, 1] for i in range(N_COEFF)]

sc_m, ev_m = coeff['in_score'], ~coeff['odd']
print('--- numbers quoted in the text ---')
print(f"score modes (odd m, n>=3): max sigma = {coeff.loc[sc_m,'std'].max():.2e}"
      f"   summed median frac. power = {coeff.loc[sc_m,'medfrac'].sum():.2e}")
print(f"  max |r| with FWHM = {coeff.loc[sc_m,'r_fwhm'].abs().max():.3f}"
      f"   max |r| with |g| = {coeff.loc[sc_m,'r_g'].abs().max():.3f}")
print(f"even-m modes             : max sigma = {coeff.loc[ev_m,'std'].max():.2e}"
      f"   summed median frac. power = {coeff.loc[ev_m,'medfrac'].sum():.4f}")
print(f"  max |r| with FWHM = {coeff.loc[ev_m,'r_fwhm'].abs().max():.3f}"
      f"   max |r| with |g| = {coeff.loc[ev_m,'r_g'].abs().max():.3f}")
print(f"ratio of max sigma, even/odd = {coeff.loc[ev_m,'std'].max()/coeff.loc[sc_m,'std'].max():.1e}")

score_atm = np.sum(bvec[:, ODD_ORDER] ** 2, axis=1) / total
print(f'\nshapelet score on the atmosphere-only population:')
print(f'  median = {np.median(score_atm):.2e}   p99 = {np.percentile(score_atm,99):.2e}'
      f'   max = {score_atm.max():.2e}')
print(f'  corr with FWHM = {np.corrcoef(score_atm, fwhm_fit)[0,1]:+.3f}'
      f'   corr with |g| = {np.corrcoef(score_atm, gmod_fit)[0,1]:+.3f}')
print(f'  -> {0.002/np.median(score_atm):.1e}x below the 0.002 LOW/MEDIUM tier boundary')

print('\nfive most atmosphere-sensitive coefficients:')
print(coeff.nlargest(5, 'std')[['idx','n','m','comp','std','medfrac','r_fwhm','r_g']]
      .to_string(index=False, float_format=lambda v: f'{v: .3e}'))

print('\nper-(n,m) summary:')
g = (coeff.groupby(['n','m'])
     .agg(sigma_max=('std','max'), medfrac=('medfrac','sum'),
          r_fwhm=('r_fwhm', lambda s: s.abs().max()),
          r_g=('r_g', lambda s: s.abs().max()),
          in_score=('in_score','first')).reset_index())
print(g.to_string(index=False, float_format=lambda v: f'{v: .3e}'))

--- numbers quoted in the text ---
score modes (odd m, n>=3): max sigma = 1.38e-16   summed median frac. power = 7.07e-33
  max |r| with FWHM = 0.086   max |r| with |g| = 0.056
even-m modes             : max sigma = 7.06e-02   summed median frac. power = 0.9996
  max |r| with FWHM = 0.453   max |r| with |g| = 0.138
ratio of max sigma, even/odd = 5.1e+14

shapelet score on the atmosphere-only population:
  median = 1.25e-32   p99 = 7.03e-32   max = 1.18e-31
  corr with FWHM = -0.041   corr with |g| = -0.005
  -> 1.6e+29x below the 0.002 LOW/MEDIUM tier boundary

five most atmosphere-sensitive coefficients:
 idx  n  m comp        std    medfrac     r_fwhm        r_g
   0  0  0       7.062e-02  9.916e-01  2.845e-01  1.383e-02
  14  4  0       2.914e-02  7.843e-03 -2.964e-01 -9.707e-03
   3  2  2   Re  2.457e-02  8.789e-05 -1.773e-02  2.159e-02
   4  2  2   Im  2.415e-02  9.206e-05 -2.375e-02 -3.475e-02
  27  6  0       5.344e-03  5.585e-05 -2.379e-01 -3.160e-02

per-(n,m) summary:
 n  m  

## Figure: coefficient response to atmospheric variation

In [ ]:
import matplotlib.patches as mpatches

colors = [mode_color(r) for _, r in coeff.iterrows()]
x = coeff['idx'].values
ticklab = [f"({r.n},{r.m}){r.comp}" for r in coeff.itertuples()]
FLOOR = 1e-18

fig, axes = plt.subplots(2, 1, figsize=(7.0, 4.8), sharex=True,
                         gridspec_kw=dict(hspace=0.10, height_ratios=[1.25, 1]))

# -- top: how much each coefficient varies across atmospheric realisations -----
ax = axes[0]
ax.bar(x, np.maximum(coeff['std'], FLOOR), color=colors, edgecolor='k', linewidth=0.3)
ax.set_yscale('log')
ax.set_ylim(FLOOR, 3e1)
ax.set_ylabel(r'$\sigma(b_{nm})$ across realisations')
ax.yaxis.set_major_locator(plt.LogLocator(base=10, numticks=8))
ax.yaxis.set_minor_locator(plt.LogLocator(base=10, subs=(), numticks=8))
ax.axhline(1e-16, color='k', ls=':', lw=0.8)
ax.text(0.995, 1.3e-16, 'double precision', ha='right', va='bottom',
        fontsize=6.5, color='k', transform=ax.get_yaxis_transform())
ax.legend(handles=[mpatches.Patch(color=C_SCORE, label='odd $m$, $n\\geq3$ (score)'),
                   mpatches.Patch(color=C_CENT, label='$n=1$ centroid'),
                   mpatches.Patch(color=C_EVEN, label='even $m$')],
          loc='upper right', frameon=False, ncol=3, fontsize=7.5,
          handlelength=1.2, columnspacing=1.2)

# -- bottom: correlation with the input atmospheric parameters ----------------
ax = axes[1]
w = 0.40
ax.bar(x - w/2, coeff['r_fwhm'], width=w, color=PALETTE[3], edgecolor='k',
       linewidth=0.3, label='FWHM')
ax.bar(x + w/2, coeff['r_g'], width=w, color=PALETTE[1], edgecolor='k',
       linewidth=0.3, label=r'$|g|$')
ax.axhline(0, color='k', lw=0.6)
ax.set_ylim(-0.55, 0.55)
ax.set_ylabel(r'Pearson $r$ with input')
ax.yaxis.set_minor_locator(MultipleLocator(0.05))
ax.yaxis.set_major_locator(MultipleLocator(0.25))
ax.legend(loc='lower right', frameon=False, ncol=2)
ax.set_xlabel(r'shapelet coefficient $(n,m)$')
ax.set_xticks(x)
ax.set_xticklabels(ticklab, rotation=90, fontsize=6)
for lab, r in zip(ax.get_xticklabels(), coeff.itertuples()):
    lab.set_color(mode_color({'in_score': r.in_score, 'centroid': r.centroid}))
ax.set_xlim(-0.8, N_COEFF - 0.2)

for a in axes:
    a.xaxis.set_minor_locator(plt.NullLocator())
    for i in x[coeff['in_score'].values]:
        a.axvspan(i - 0.5, i + 0.5, color=C_SCORE, alpha=0.08, lw=0, zorder=0)

fig.savefig(FIG / 'atm_coeff_variation.pdf')
print('wrote', FIG / 'atm_coeff_variation.pdf')